# Temporal ALLD Data Generation Sandbox (V2)

This notebook demonstrates how clean speech is combined with random degradations: 
1. **Additive Noise** (ESC-50)
2. **Packet Loss** (Temporal Muting)
3. **Clipping Distortion** (Temporal Hard-Clipping)

It allows you to listen to random samples and inspect the exact metadata timestamps generated.

In [ ]:
import torch
import torchaudio
import IPython.display as ipd
import matplotlib.pyplot as plt
import random

from asa.generate_temporal_data import load_clean_speech_dataset, load_noise_dataset, overlay_noise, apply_packet_loss, apply_clipping

In [ ]:
# Load iterators (streaming)
clean_iter = load_clean_speech_dataset(split="validation", streaming=True)
noise_iter = load_noise_dataset(split="train", streaming=True)

In [ ]:
# 1. Fetch random samples
clean_sample = next(clean_iter)

clean_audio = clean_sample["audio"]
clean_tensor = torch.tensor(clean_audio["array"]).view(1, -1).float()
sr = 16000
if clean_audio["sampling_rate"] != sr:
    clean_tensor = torchaudio.functional.resample(clean_tensor, clean_audio["sampling_rate"], sr)

# Randomly choose degradation
deg_choice = random.choice(["noise", "packet_loss", "clipping"])

if deg_choice == "noise":
    noise_sample = next(noise_iter)
    noise_audio = noise_sample["audio"]
    noise_category = noise_sample.get("category", "noise")
    noise_tensor = torch.tensor(noise_audio["array"]).view(1, -1).float()
    mixed_waveform, start_time, end_time = overlay_noise(
        clean_tensor, sr, noise_tensor, noise_audio["sampling_rate"], target_sr=sr
    )
    print(f"Degradation: Noise ({noise_category})")
elif deg_choice == "packet_loss":
    mixed_waveform, start_time, end_time = apply_packet_loss(clean_tensor, sr)
    print("Degradation: Packet Loss")
else:
    mixed_waveform, start_time, end_time = apply_clipping(clean_tensor, sr, threshold=random.uniform(0.05, 0.1))
    print("Degradation: Clipping Distortion")

In [ ]:
# 2. Visualize
print(f"Metadata:\n- Start: {start_time:.2f}s\n- End: {end_time:.2f}s")

plt.figure(figsize=(10, 3))
plt.plot(mixed_waveform.t().numpy())
plt.axvline(x=start_time * 16000, color='r', linestyle='--', label="Start degradation")
plt.axvline(x=end_time * 16000, color='g', linestyle='--', label="End degradation")
plt.legend()
plt.title("Degraded Waveform with Temporal Bounds")
plt.show()

# 3. Play Audio
ipd.display(ipd.Audio(mixed_waveform.numpy(), rate=16000))